In [ ]:
from google.colab import drive
import os

# 1. Mount the Google Drive
drive.mount('/content/drive')

# 2. Define the exact path to the shared project folder
PROJECT_ROOT = '/content/drive/MyDrive/MNIST-Project/'

# 3. Automatically create the required sub-folders if they don't exist yet
os.makedirs(os.path.join(PROJECT_ROOT, 'data'), exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, 'models'), exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, 'results'), exist_ok=True)

print("Drive successfully mounted and project directories are ready!")

Mounted at /content/drive
Drive successfully mounted and project directories are ready!


In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Input
from tensorflow.keras.models import Sequential

print("Loading 2D data for CNN baselines...")
data_dir = os.path.join(PROJECT_ROOT, 'data')

# Load Training Splits (and reshape to add the grayscale channel)
x_train_5k = np.load(os.path.join(data_dir, 'x_train_5k_2d.npy')).reshape(-1, 28, 28, 1)
y_train_5k = np.load(os.path.join(data_dir, 'y_train_5k.npy'))

x_train_15k = np.load(os.path.join(data_dir, 'x_train_15k_2d.npy')).reshape(-1, 28, 28, 1)
y_train_15k = np.load(os.path.join(data_dir, 'y_train_15k.npy'))

# Load Testing Data
x_test = np.load(os.path.join(data_dir, 'x_test_2d.npy')).reshape(-1, 28, 28, 1)
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

print("Data loaded successfully!")

Loading 2D data for CNN baselines...
Data loaded successfully!


In [ ]:
def build_basic_cnn():
    model = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, kernel_size=(3, 3), activation='relu'),
        Flatten(),
        Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
print("--- Training Baseline CNN (5,000 Images) ---")
cnn_5k = build_basic_cnn()

start_time = time.time()
# We train for 5 epochs to get a quick baseline
history_5k = cnn_5k.fit(x_train_5k, y_train_5k, epochs=5, validation_data=(x_test, y_test), batch_size=32)
time_5k = time.time() - start_time

print(f"Training Time (5k): {time_5k:.2f} seconds")
print(f"Final Test Accuracy (5k): {history_5k.history['val_accuracy'][-1] * 100:.2f}%")

--- Training Baseline CNN (5,000 Images) ---
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.8336 - loss: 0.5705 - val_accuracy: 0.9092 - val_loss: 0.3108
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9362 - loss: 0.2248 - val_accuracy: 0.9399 - val_loss: 0.2078
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9628 - loss: 0.1321 - val_accuracy: 0.9474 - val_loss: 0.1819
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9796 - loss: 0.0802 - val_accuracy: 0.9554 - val_loss: 0.1565
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9896 - loss: 0.0491 - val_accuracy: 0.9582 - val_loss: 0.1459
Training Time (5k): 9.14 seconds
Final Test Accuracy (5k): 95.82%


In [ ]:
print("--- Training Baseline CNN (15,000 Images) ---")
cnn_15k = build_basic_cnn()

start_time = time.time()
history_15k = cnn_15k.fit(x_train_15k, y_train_15k, epochs=5, validation_data=(x_test, y_test), batch_size=32)
time_15k = time.time() - start_time

print(f"Training Time (15k): {time_15k:.2f} seconds")
print(f"Final Test Accuracy (15k): {history_15k.history['val_accuracy'][-1] * 100:.2f}%")

--- Training Baseline CNN (15,000 Images) ---
Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8990 - loss: 0.3462 - val_accuracy: 0.9539 - val_loss: 0.1690
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9655 - loss: 0.1257 - val_accuracy: 0.9651 - val_loss: 0.1193
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9793 - loss: 0.0759 - val_accuracy: 0.9687 - val_loss: 0.1026
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9856 - loss: 0.0512 - val_accuracy: 0.9705 - val_loss: 0.1004
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9901 - loss: 0.0352 - val_accuracy: 0.9725 - val_loss: 0.0932
Training Time (15k): 13.71 seconds
Final Test Accuracy (15k): 97.25%


In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

data_dir = os.path.join(PROJECT_ROOT, 'data')
models_dir = os.path.join(PROJECT_ROOT, 'models')

# Load the FULL 60k 2D Dataset
print("Loading FULL 60k 2D data for Advanced CNN...")
# Reshape to add the color channel for the CNN
x_train_full_2d = np.load(os.path.join(data_dir, 'x_train_full_2d.npy')).reshape(-1, 28, 28, 1)
y_train_full = np.load(os.path.join(data_dir, 'y_train_full.npy'))

x_test_2d = np.load(os.path.join(data_dir, 'x_test_2d.npy')).reshape(-1, 28, 28, 1)
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

print(f"Data loaded successfully! Training shape: {x_train_full_2d.shape}")

Loading FULL 60k 2D data for Advanced CNN...
Data loaded successfully! Training shape: (60000, 28, 28, 1)


In [ ]:
def build_advanced_cnn():
    model = Sequential([
        Input(shape=(28, 28, 1)),

        # Block 1: Feature Extraction
        Conv2D(32, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)), # Shrinks 28x28 to 14x14

        # Block 2: Deep Feature Extraction
        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)), # Shrinks 14x14 to 7x7

        # Bridge to Decision Making
        Flatten(),
        Dense(128, activation='relu'),

        # Regularization
        Dropout(0.5),

        # Output Layer
        Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print("Advanced CNN Architecture defined!")

Advanced CNN Architecture defined!


In [ ]:
print("--- Training Advanced CNN on FULL 60k Dataset ---")
advanced_cnn = build_advanced_cnn()

start_time = time.time()
# We are passing the full dataset here
history_advanced = advanced_cnn.fit(
    x_train_full_2d, y_train_full,
    epochs=10,
    batch_size=64,
    validation_data=(x_test_2d, y_test)
)
training_time = time.time() - start_time

print(f"\nFinal Training Time: {training_time:.2f} seconds")
print(f"Final Test Accuracy: {history_advanced.history['val_accuracy'][-1] * 100:.2f}%")

final_model_path = os.path.join(models_dir, 'advanced_cnn.keras')
advanced_cnn.save(final_model_path)
print(f"Final Model successfully saved to: {final_model_path}")

--- Training Advanced CNN on FULL 60k Dataset ---
Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.9309 - loss: 0.2238 - val_accuracy: 0.9834 - val_loss: 0.0506
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9754 - loss: 0.0826 - val_accuracy: 0.9884 - val_loss: 0.0379
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9826 - loss: 0.0600 - val_accuracy: 0.9897 - val_loss: 0.0301
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9855 - loss: 0.0472 - val_accuracy: 0.9912 - val_loss: 0.0272
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9875 - loss: 0.0420 - val_accuracy: 0.9920 - val_loss: 0.0259
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9902 - loss: 0.0330 - val_accuracy: 0.9914 - val_loss: 0.0266
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9908 - loss: 0.0301 - val_accuracy: 0.9920 - val_loss: 0.0241
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - 